# 01 Data audit

What is in each file, and what quirks do we need to handle? Timestamps are UTC; days and run times are UK local.

## Init

In [1]:
import sys
sys.path[:0] = ["notebooks", "."]  # _common imports from repo root or notebooks/

import pandas as pd
import matplotlib.pyplot as plt

from _common import load, load_all, to_uk, settlement_day, UK_TZ

pd.set_option("display.width", 160)
pd.set_option("display.max_columns", 30)

In [2]:
data = load_all()
for name, df in data.items():
    print(f"{name:32s} {df.shape}")

imbalance_price                  (35039, 2)
demand_forecast_elexon           (2055958, 4)
wind_forecast_elexon             (422741, 4)
wind_solar_forecast_embedded     (2509386, 7)
margin_half_hourly               (961662, 5)
margin_daily                     (1458, 15)


## 1. Overview per table
Shape, period range, nulls and exact duplicate rows.

In [3]:
rows = []
for name, df in data.items():
    tcol = "period_start" if "period_start" in df else "day"
    rows.append(dict(table=name, rows=len(df), first=str(df[tcol].min())[:16], last=str(df[tcol].max())[:16],
                     null_cells=int(df.isna().sum().sum()), dup_rows=int(df.duplicated().sum())))
pd.DataFrame(rows).set_index("table")

,rows,first,last,null_cells,dup_rows
table,,,,,
imbalance_price,35039,2024-09-30 23:00,2026-09-30 22:30,0,0
demand_forecast_elexon,2055958,2024-09-27 23:30,2026-10-02 03:30,0,0
wind_forecast_elexon,422741,2024-09-27 20:00,2026-10-02 20:00,0,0
wind_solar_forecast_embedded,2509386,2024-09-27 23:00,2026-10-03 22:30,2509386,0
margin_half_hourly,961662,2025-10-09 16:00,2026-10-02 03:30,0,0
margin_daily,1458,2024-09-30,2026-10-03,1536,0


## 2. Prices (`imbalance_price`)
One row per half-hour. Check the grid for gaps/duplicates and the per-UK-day period counts (46/48/50).

In [4]:
p = data["imbalance_price"].set_index("period_start")["price_gbp_per_mwh"]
full = pd.date_range(p.index.min(), p.index.max(), freq="30min")
missing = full.difference(p.index)
print("duplicate timestamps:", p.index.duplicated().sum())
print("missing half-hours in range:", len(missing))
print(pd.Series(missing).dt.date.value_counts().sort_index().head(15))
print(p.describe().round(2))

duplicate timestamps: 0
missing half-hours in range: 1
2026-01-18    1
Name: count, dtype: int64
count    35039.00
mean        89.72
std         66.84
min        -97.92
25%         64.63
50%         90.90
75%        117.00
max       2900.00
Name: price_gbp_per_mwh, dtype: float64


In [5]:
n = p.groupby(settlement_day(p.index.to_series())).size()
print(n.value_counts().sort_index())
print("days with != 48 periods:"); print(n[n != 48])

price_gbp_per_mwh
46      2
47      1
48    725
50      2
Name: count, dtype: int64
days with != 48 periods:
period_start
2024-10-27    50
2025-03-30    46
2025-10-26    50
2026-01-18    47
2026-03-29    46
Name: price_gbp_per_mwh, dtype: int64


## 3. Forecast tables: vintages
How many vintages per target period, how far ahead they are issued, and the issue-to-receipt delay (relevant for what was *known* at the cutoff).

In [6]:
fc = ["demand_forecast_elexon","wind_forecast_elexon","wind_solar_forecast_embedded","margin_half_hourly"]
rows=[]
for name in fc:
    df = data[name]
    per = df.groupby("period_start").size()
    ahead = (df["period_start"] - df["issued_at"]).dt.total_seconds()/3600
    delay = (df["received_at"] - df["issued_at"]).dt.total_seconds()/60
    rows.append(dict(table=name, periods=per.size, vint_median=per.median(), vint_min=per.min(), vint_max=per.max(),
                     ahead_h_min=ahead.min().round(1), ahead_h_med=ahead.median().round(1), ahead_h_max=ahead.max().round(1),
                     recv_delay_min_med=delay.median().round(1), recv_delay_min_max=delay.max().round(1),
                     recv_before_issue=int((delay<0).sum())))
pd.DataFrame(rows).set_index("table")

,periods,vint_median,vint_min,vint_max,ahead_h_min,ahead_h_med,ahead_h_max,recv_delay_min_med,recv_delay_min_max,recv_before_issue
table,,,,,,,,,,
demand_forecast_elexon,35161,58.0,1,84,-34.0,14.7,44.8,0.8,3535.9,0
wind_forecast_elexon,17641,24.0,1,32,-27.5,20.5,66.5,0.5,614.4,0
wind_solar_forecast_embedded,35287,72.0,1,78,-0.9,35.8,72.0,NaN,NaN,0
margin_half_hourly,17160,56.0,1,82,-5.0,14.9,41.4,0.3,995.6,160


In [7]:
# period coverage vs the price grid, and rows per period (hourly tables show as 1 period per 2 prices)
pgrid = set(p.index)
for name in fc:
    ps = set(data[name]["period_start"].unique())
    print(f"{name:32s} periods={len(ps):6d}  price periods covered={len(pgrid & ps)/len(pgrid):.1%}  "
          f"first={min(ps):%Y-%m-%d}  last={max(ps):%Y-%m-%d}")

demand_forecast_elexon           periods= 35161  price periods covered=99.8%  first=2024-09-27  last=2026-10-02
wind_forecast_elexon             periods= 17641  price periods covered=50.0%  first=2024-09-27  last=2026-10-02


wind_solar_forecast_embedded     periods= 35287  price periods covered=99.9%  first=2024-09-27  last=2026-10-03
margin_half_hourly               periods= 17160  price periods covered=48.8%  first=2025-10-09  last=2026-10-02


In [8]:
# sub-hourly granularity check for wind_forecast_elexon (expected hourly)
w = data["wind_forecast_elexon"]["period_start"].drop_duplicates().sort_values()
print(w.diff().value_counts().head(5))

period_start
0 days 01:00:00    17640
Name: count, dtype: int64


## 4. `margin_daily`
`issued_at` is date-like (no time of day?) and `day` is the target date.

In [9]:
m = data["margin_daily"]
print(m[["issued_at","received_at","day"]].head(5))
print("issued_at time-of-day values:", m["issued_at"].dt.time.nunique(), "| unique days:", m["day"].nunique(), "| rows/day:", len(m)/m["day"].nunique())
lead = (pd.to_datetime(m["day"]).dt.tz_localize("UTC") - m["issued_at"]).dt.days
print(lead.value_counts().sort_index())
print(m.describe().T[["count","mean","min","max"]].round(0))

                  issued_at received_at         day
0 2024-09-27 23:00:00+00:00         NaT  2024-09-30
1 2024-09-27 23:00:00+00:00         NaT  2024-10-01
2 2024-09-28 23:00:00+00:00         NaT  2024-10-01
3 2024-09-28 23:00:00+00:00         NaT  2024-10-02
4 2024-09-29 23:00:00+00:00         NaT  2024-10-02
issued_at time-of-day values: 2 | unique days: 734 | rows/day: 1.986376021798365
2    732
3    726
Name: count, dtype: int64
                                            count     mean      min      max
peak_demand_forecast_mw                    1458.0  32670.0  21437.0  45934.0
minimum_demand_forecast_mw                 1458.0  19933.0  11977.0  28141.0
generator_availability_mw                  1458.0  46565.0  32222.0  61752.0
generation_availability_margin_mw          1458.0  13895.0  -1886.0  31822.0
operating_margin_requirement_mw            1458.0   5442.0   2518.0  15261.0
national_surplus_mw                        1458.0   8453.0  -4498.0  20836.0
max_interconnector_impor

## 5. Availability at the cutoff
For each target day D, at D-1 07:00 / 09:00 UK: what share of D's periods have at least one vintage already available? Availability time = `received_at`, falling back to `issued_at` where there is no `received_at` (embedded forecast). Share is averaged over target days.

In [10]:
def avail(df):
    return df["received_at"].fillna(df["issued_at"]) if "received_at" in df else df["issued_at"]

def coverage(name, hour):
    df = data[name]
    day = pd.to_datetime(settlement_day(df["period_start"]))
    cutoff = ((day - pd.Timedelta(days=1)).dt.tz_localize(UK_TZ) + pd.Timedelta(hours=hour)).dt.tz_convert("UTC")
    ok = df[avail(df) <= cutoff]
    n_ok = ok.groupby(day[ok.index]).period_start.nunique()
    n_all = df.groupby(day).period_start.nunique()
    return (n_ok.reindex(n_all.index).fillna(0) / n_all).mean()

pd.DataFrame({f"{h:02d}:00": {n: coverage(n, h) for n in fc} for h in (7, 9)}).round(3)

,07:00,09:00
demand_forecast_elexon,0.209,0.991
wind_forecast_elexon,0.993,0.994
wind_solar_forecast_embedded,0.996,0.996
margin_half_hourly,0.209,0.209


In [11]:
# When does the first vintage covering each period become available, relative to the start of D-1 (UK)?
# i.e. hours after D-1 00:00 UK. 07:00 -> 7, 09:00 -> 9.
rows = {}
for name in fc:
    df = data[name]
    day = pd.to_datetime(settlement_day(df["period_start"]))
    d1 = (day - pd.Timedelta(days=1)).dt.tz_localize(UK_TZ).dt.tz_convert("UTC")
    first = (avail(df) - d1).dt.total_seconds() / 3600
    s = first.groupby(df["period_start"]).min()
    rows[name] = s.describe(percentiles=[.05, .5, .95]).round(1)
pd.DataFrame(rows)

,demand_forecast_elexon,wind_forecast_elexon,wind_solar_forecast_embedded,margin_half_hourly
count,35161.0,17641.0,35287.0,17160.0
mean,3.8,-18.0,-35.4,7.2
std,9.8,7.8,7.9,9.8
min,-15.2,-21.5,-48.8,-11.9
5%,-15.2,-21.5,-46.8,-11.9
50%,8.8,-19.5,-35.8,12.1
95%,8.8,4.5,-24.8,12.1
max,56.7,80.7,35.1,40.1


In [12]:
# Example: demand_forecast_elexon for D = 2026-03-10, vintages received per hour (UK) that cover D
D = pd.Timestamp("2026-03-10", tz=UK_TZ)
df = data["demand_forecast_elexon"]
sub = df[(df.period_start >= D.tz_convert("UTC")) & (df.period_start < (D + pd.Timedelta(days=1)).tz_convert("UTC"))]
print(sub.groupby(to_uk(sub.received_at).dt.floor("h")).period_start.nunique().loc["2026-03-09 06:00":"2026-03-09 12:00"])

received_at
2026-03-09 06:00:00+00:00    10
2026-03-09 07:00:00+00:00    10
2026-03-09 08:00:00+00:00    48
2026-03-09 09:00:00+00:00    10
2026-03-09 10:00:00+00:00    10
2026-03-09 11:00:00+00:00    48
2026-03-09 12:00:00+00:00    48
Name: period_start, dtype: int64


In [13]:
# margin_half_hourly: rows received before they were issued
mh = data["margin_half_hourly"]
bad = mh[mh.received_at < mh.issued_at]
print(len(bad), "rows;", bad.issued_at.nunique(), "distinct issued_at;", str(bad.issued_at.min()), "->", str(bad.issued_at.max()))
print("max price:", p.max(), "at", p.idxmax(), "| prices > 500:", (p > 500).sum(), "| negatives:", (p < 0).sum())

160 rows; 2 distinct issued_at; 2026-06-04 16:00:13+00:00 -> 2026-07-07 17:00:18+00:00
max price: 2900.0 at 2025-01-08 15:00:00+00:00 | prices > 500: 29 | negatives: 1931


## Findings

**Prices**
- 35,039 rows, 2024-10-01 to 2026-09-30 (UK days), no nulls or duplicate timestamps. One half-hour missing (2026-01-18, a 47-period day); clock-change days have 46/50 periods as expected (2024-10-27, 2025-03-30, 2025-10-26, 2026-03-29).
- Heavy tails: max 2,900 at 2025-01-08 15:00 UTC, 29 periods above 500, 1,931 negative. Median about 91.

**Forecast tables (all have many vintages per target period)**
- `wind_forecast_elexon` is hourly (covers 50% of price timestamps); `margin_half_hourly` only starts 2025-10-09 (about 1 year).
- `wind_solar_forecast_embedded` has no `received_at` (all null): use `issued_at` as the availability time. `margin_daily.received_at` is also all null; its `issued_at` is a midnight-UK stamp issued 2-3 days ahead.
- `margin_half_hourly`: 160 rows are received about 2h *before* `issued_at` (2 batches, June and July 2026), a timestamp glitch. Use `received_at` as the availability time anyway; it is the more conservative.

**What is known at the cutoff (the key finding)**
- Wind and embedded wind/solar forecasts are available for ~99% of day D's periods by 07:00 D-1.
- **`demand_forecast_elexon`: only 21% of D's periods (about the first 10, overnight) have any vintage at 07:00. The full-day forecast arrives around 08:48 D-1, so the 09:00 run sees it (99%) but the 07:00 run does not.**
- **`margin_half_hourly`: the full-day vintage arrives about 12:06 D-1, after both cutoffs, so it is only available for ~21% of periods. It adds almost nothing at either run and only exists for one year.**
- So the 07:00 and 09:00 runs genuinely differ in available information; the 07:00 model cannot use same-day-ahead demand for most of the day.

**To handle in 03 (point-in-time features)**
- Use `received_at` (fallback `issued_at`) <= cutoff, taking the latest vintage per period; handle periods with no vintage explicitly (missing, not leaked).
- Convert D-1 07:00/09:00 UK to UTC per day (DST).
- Decide how to treat the missing 2026-01-18 period and the hourly wind grid.
